In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime
import sqlite3

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [11]:
# %%
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

BASE_URL = "https://european-union.europa.eu/news-and-events/news-and-stories_en"
print("✅ Scraper da União Europeia (News & Stories) pronto!")

✅ Scraper da União Europeia (News & Stories) pronto!


In [12]:
DATABASE_NAME = "internet_governance_news.db"

def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            author TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela 'articles' prontos!")

create_database()


✅ Banco e tabela 'articles' prontos!


In [13]:
def insert_article(title, date, author, url, source):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, author, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, author, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

In [14]:
# %%
def load_articles_from_db():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT *
        FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles_from_db()
display(df_db.head())
print(f"📦 Total no banco: {len(df_db)} registros")


,id,title,date,author,url,source
0,3090,EDPB brings clarity to data processing for sci...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
1,3091,Enhancing compliance and consistency: EDPB ado...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/enha...,EDPB
2,3092,EDPB annual report 2025: supporting stakeholde...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
3,3093,EDPB conference on cross-regulatory cooperatio...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
4,3094,EDPB and EDPS support strengthening EU’s cyber...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB


📦 Total no banco: 3601 registros


In [15]:
# %%
def montar_url(pagina):
    """Monta a URL paginada da EU News (page=0 é a primeira)."""
    if pagina == 0:
        return BASE_URL
    return f"{BASE_URL}?page={pagina}"

def extrair_paragrafos(url):
    """Acessa a página da notícia e extrai os parágrafos principais."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        ps = soup.find_all("p")
        textos = [
            p.get_text(strip=True)
            for p in ps
            if len(p.get_text(strip=True).split()) > 10
        ]
        return textos[:5] if textos else ["NA"]
    except:
        return ["NA"]

In [16]:
# %%
noticias = []
TOTAL_PAGES = 5  # paginas 0..49 (~993 noticias, ~20 por pagina), ajustar conforme o necessario, sao 50 paginas totais

for pagina in range(0, TOTAL_PAGES):
    url = montar_url(pagina)
    print(f"\U0001f4c4 Coletando pagina {pagina + 1}/{TOTAL_PAGES}: {url}")

    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            print(f"\u26a0\ufe0f Erro HTTP {r.status_code} \u2014 parando.")
            break
    except Exception as e:
        print(f"\u26a0\ufe0f Erro de conex\u00e3o: {e}")
        break

    soup = BeautifulSoup(r.text, "html.parser")

    # ===== Seletores para european-union.europa.eu (ECL framework)
    # As noticias ficam em <article class="ecl-content-item">.
    # Cada item tem: tipo e data em .ecl-content-block__primary-meta-item,
    # titulo em .ecl-content-block__title a, resumo em .ecl-content-block__description.
    # A data tambem esta em <time datetime="...">.
    itens = soup.select(".ecl-content-item")

    count_page = 0
    for item in itens:
        # Extrai data via <time> tag
        time_tag = item.select_one("time")
        if time_tag:
            data_raw = time_tag.get_text(strip=True)
        else:
            data_raw = "NA"

        # Extrai titulo e link via .ecl-content-block__title a
        link_tag = item.select_one(".ecl-content-block__title a")
        if not link_tag:
            continue

        titulo = link_tag.get_text(strip=True)
        link = link_tag.get("href", "")

        # Ignora links de navegacao/menu (muito curtos)
        if len(titulo) < 15:
            continue

        # Complementa URL relativa
        if link.startswith("/"):
            link = f"https://european-union.europa.eu{link}"

        # Extrai paragrafos da noticia
        paragrafos = extrair_paragrafos(link)

        # Acumula
        noticias.append({
            "titulo": titulo,
            "data": data_raw,
            "link": link,
            "paragrafos": " || ".join(paragrafos),
            "fonte": "Uni\u00e3o Europeia"
        })

        # Grava no banco
        insert_article(
            title=titulo,
            date=data_raw,
            author="European Union",
            url=link,
            source="Uni\u00e3o Europeia"
        )

        count_page += 1

    print(f"   {count_page} noticias extraidas")

    if count_page == 0:
        print("\U0001f3c1 Nenhuma noticia encontrada \u2014 fim da coleta.")
        break

    time.sleep(1)

print(f"\n\u2705 Total coletado: {len(noticias)} noticias")

df_eu = pd.DataFrame(noticias)
display(df_eu.head())

📄 Coletando pagina 1/5: https://european-union.europa.eu/news-and-events/news-and-stories_en
   20 noticias extraidas
📄 Coletando pagina 2/5: https://european-union.europa.eu/news-and-events/news-and-stories_en?page=1
   20 noticias extraidas
📄 Coletando pagina 3/5: https://european-union.europa.eu/news-and-events/news-and-stories_en?page=2
   20 noticias extraidas
📄 Coletando pagina 4/5: https://european-union.europa.eu/news-and-events/news-and-stories_en?page=3
   20 noticias extraidas
📄 Coletando pagina 5/5: https://european-union.europa.eu/news-and-events/news-and-stories_en?page=4
   20 noticias extraidas

✅ Total coletado: 100 noticias


,titulo,data,link,paragrafos,fonte
0,EU history: celebrating 75 years since the Tre...,17 April 2026,https://commission.europa.eu/news-and-media/ne...,On April 18 we celebrate 75 years since the si...,União Europeia
1,Over 10 million workers trained under the EU’s...,16 April 2026,https://ec.europa.eu/commission/presscorner/de...,NA,União Europeia
2,European age verification app to keep children...,16 April 2026,https://commission.europa.eu/news-and-media/ne...,The European Commission has announced that a n...,União Europeia
3,New report looks at how algorithms fuel polari...,16 April 2026,https://joint-research-centre.ec.europa.eu/jrc...,"Overly curated, attention-grabbing content can...",União Europeia
4,EU provides over €800 million in response to c...,16 April 2026,https://ec.europa.eu/commission/presscorner/de...,NA,União Europeia


In [17]:
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT * FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles()
print(f"📦 Total no banco: {len(df_db)} registros")
display(df_db.head(20))

📦 Total no banco: 3700 registros


,id,title,date,author,url,source
0,3090,EDPB brings clarity to data processing for sci...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
1,3091,Enhancing compliance and consistency: EDPB ado...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/enha...,EDPB
2,3092,EDPB annual report 2025: supporting stakeholde...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
3,3093,EDPB conference on cross-regulatory cooperatio...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
4,3094,EDPB and EDPS support strengthening EU’s cyber...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
5,3095,CEF 2026: EDPB launches coordinated enforcemen...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/cef-...,EDPB
6,3096,EDPB and EDPS support harmonisation of clinica...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
7,3097,Stakeholder event on political advertising: ag...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/stak...,EDPB
8,3098,Conference on cross-regulatory cooperation in ...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/conf...,EDPB
9,3099,AI-generated imagery and protection of privacy...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/ai-g...,EDPB


In [18]:
keywords = ['digital', 'internet', 'IA', 'tecnologia', 'dados', 'privacidade',
            'data', 'privacy', 'AI', 'cybersecurity', 'online', 'algorithm',
            'artificial intelligence', 'protection', 'regulation']
pattern = r'|'.join(keywords)

df_filt = df_eu[
    df_eu['titulo'].str.contains(pattern, case=False, na=False, regex=True) |
    df_eu['paragrafos'].str.contains(pattern, case=False, na=False, regex=True)
].copy()

print(f"{len(df_filt)} notícias filtradas (de {len(df_eu)})")
display(df_filt.head())

80 notícias filtradas (de 100)


,titulo,data,link,paragrafos,fonte
0,EU history: celebrating 75 years since the Tre...,17 April 2026,https://commission.europa.eu/news-and-media/ne...,On April 18 we celebrate 75 years since the si...,União Europeia
1,Over 10 million workers trained under the EU’s...,16 April 2026,https://ec.europa.eu/commission/presscorner/de...,NA,União Europeia
2,European age verification app to keep children...,16 April 2026,https://commission.europa.eu/news-and-media/ne...,The European Commission has announced that a n...,União Europeia
3,New report looks at how algorithms fuel polari...,16 April 2026,https://joint-research-centre.ec.europa.eu/jrc...,"Overly curated, attention-grabbing content can...",União Europeia
5,‘Deaf’ wins 2026 LUX Audience Award,15 April 2026,https://www.europarl.europa.eu/news/en/press-r...,"Chosen jointly by EU citizens and MEPs, Spanis...",União Europeia


In [19]:
def plot_charts(df):
    if df.empty:
        print("❌ Sem dados para gráficos")
        return

    # ------------------------------
    # Top 15
    # ------------------------------
    top15 = df.head(15).copy()
    top15['rank'] = range(1, len(top15) + 1)

    fig1 = px.bar(
        top15,
        x='rank',
        y='title',
        orientation='h',
        title='Top 15 Notícias – Internet Governance'
    )
    fig1.update_layout(height=600)
    fig1.show()

    # ------------------------------
    # Pizza por Fonte (BANCO)
    # ------------------------------
    source_count = df["source"].value_counts().reset_index()
    source_count.columns = ["source", "count"]

    fig2 = px.pie(
        source_count,
        names="source",
        values="count",
        title="Distribuição por Fonte"
    )
    fig2.show()

    # ------------------------------
    # Nuvem de Palavras
    # ------------------------------
    text = ' '.join(df['title'].astype(str)).lower()
    words = re.findall(r'\b\w{4,}\b', text)

    wc = (
        pd.Series(words)
        .value_counts()
        .head(20)
        .reset_index()
    )
    wc.columns = ['palavra', 'freq']

    fig3 = px.treemap(
        wc,
        path=['palavra'],
        values='freq',
        title='Nuvem de Palavras – Títulos'
    )
    fig3.show()

In [21]:
plot_charts(df_db)